In [265]:
# To install: pip install tavily-python
from tavily import TavilyClient
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
load_dotenv(override=True)
import re
from random import sample
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime, timedelta
from tqdm import tqdm
import json
import requests

In [219]:
tavily_client = TavilyClient(os.getenv("TAVILY_KEY"))

openai_client = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # LM Studio doesn't require a real API key,
    model="qwen-lmstudio",
    temperature=0,
)

In [220]:
_MAX_RESULTS = 3

In [221]:
search_response = tavily_client.search(
    query="What are the kid friendly activities in Singapore happening in the coming weekend?",
    time_range="week",
    country="Singapore",
    max_results=_MAX_RESULTS
)


In [222]:
class EventFormat(BaseModel):
    """Each event."""
    event_title: str = Field(description="A short title for the event.")
    location: str = Field(description="Event location")
    date: str = Field(description="Event date and time")
    event_details: str = Field(description="Summary of what the event is about.")
    age_level: Optional[str] = Field(None, description="age group relevant for the event")
    url: Optional[str] = None

class EventList(BaseModel):
    """List of events"""
    event_list: List[EventFormat]

In [223]:
# 3. Bind structured output
structured_llm = openai_client.with_structured_output(EventList)


In [224]:
def get_todays_date():
    return datetime.now().date()

get_todays_date()


def is_weekend(date):
    return date.weekday() in (5, 6)  # Saturday and Sunday are considered weekends

def get_weekend_dates_in_range(start_date):
    """Returns a list of all weekend dates within a given date range."""
    weekend_dates = []
    current_date = start_date
    while current_date <= start_date + timedelta(days=7):
        if is_weekend(current_date):
            weekend_dates.append(current_date.strftime(format="%d/%m/%Y"))
        current_date += timedelta(days=1)
    return weekend_dates

allowed_dates = get_weekend_dates_in_range(get_todays_date())

allowed_dates 
# format the dates nicely as 6th sept 2025


['06/09/2025', '07/09/2025']

In [ ]:
all_urls = [x["url"] for x in search_response["results"]]

extracted_texts = tavily_client.extract(
    urls=all_urls,
    # format="text"
)

all_events = []

for i, url in enumerate(all_urls):
    try:
        print(f"Processing {i+1}/{len(all_urls)}: {url}")
        text_from_site = extracted_texts["results"][i]["raw_content"][:]

        chunked_texts = RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
        ).split_text(text_from_site)
        # lets iterate through the events and print them out nicely

        for text in tqdm(chunked_texts[:]):
            output = structured_llm.invoke(
                f"""
                f"What are some kid-friendly activities happening in Singapore this weekend - on these dates: {', '.join(allowed_dates)}. 
                If no events match, return an empty list. Have a preference for events that are free or low cost and are outdoor.
                Do not include events that are generic - like museums or zoos that are always open.
                Answer based on the following content only: {text}, """
            )
            if output.event_list:
                for event in output.event_list:
                    event.url = url
                    all_events.append(event)
    except Exception as e:
        print(f"Error processing {url}: {e}")
        continue



Processing 1/3: https://www.sassymamasg.com/play-weekend-planner-fun-activities-events-kids/


100%|██████████| 17/17 [02:31<00:00,  8.91s/it]


Processing 2/3: https://www.bykido.com/blogs/events-and-activities/top-things-to-do-and-places-to-go-with-kids-this-weekend-in-singapore?srsltid=AfmBOoqWgLrta2lAmmYi4BQgBgvgV9FpmrQuQOMpZkL6MmtKwtiwps5G


100%|██████████| 44/44 [03:13<00:00,  4.40s/it]


Processing 3/3: https://www.sunnycitykids.com/blog/kids-activities-this-week-in-singapore


100%|██████████| 69/69 [07:31<00:00,  6.54s/it]


In [ ]:

def dedup_with_llm(events: List[EventFormat]) -> List[EventFormat]:
    events_json = [e.model_dump() for e in events]
    sampled_events = sample(events_json, min(len(events_json), 40)) # sample max 20 events to avoid token limits
    prompt = f"""
    You are given a list of event objects in JSON. Some may be duplicates with slight wording differences.
    First, deduplicate them, keeping the most complete version of each unique event.
    And then, choose ones that are super interesting, kids would love and unique. Suggest me only 5 max of these events. 
    Return only valid JSON: a list of event objects.
    
    Input:
    {json.dumps(sampled_events, indent=2)}
    """
    
    return structured_llm.invoke(prompt)

In [243]:
deduped_events = dedup_with_llm(all_events)

In [245]:
for e in deduped_events.event_list:
    print(e)

event_title='Propnex Family Zone Outdoor Playground at Gardens by the Bay' location='Gardens by the Bay, Propnex Family Zone, near Active Garden and Waterfront Plaza' date='Till 31 Dec 2025' event_details='Free outdoor playground featuring climbing towers, suspension bridges, sand pit, playhouse, market store, and obstacle courses. Beautiful backdrop with Marina Bay Sands.' age_level='Kids' url='https://www.bykido.com/blogs/events-and-activities/top-things-to-do-and-places-to-go-with-kids-this-weekend-in-singapore?srsltid=AfmBOoqWgLrta2lAmmYi4BQgBgvgV9FpmrQuQOMpZkL6MmtKwtiwps5G'
event_title='*The Great Reef Rescue*: A Fun & Free Conservation Adventure for Families' location='Not specified in the content provided' date='06/09/2025 and 07/09/2025' event_details='A free conservation adventure that encourages families to participate in a fun activity aimed at raising awareness about reef conservation.' age_level='All ages' url='https://www.sunnycitykids.com/blog/kids-activities-this-week-i

In [277]:
# pydantic objects to markdown
def events_to_markdown(events: List[EventFormat]) -> str:
    lines = []
    for idx, ev in enumerate(events, start=1):
        entry = (
            f"{idx}. {ev.event_title}*\n"
            f"📍 {ev.location}\n"
            f"🗓 {ev.date}\n"
            f"{ev.event_details}\n"
        )
        if ev.age_level:
            entry += f"👶 Age group: {ev.age_level}\n"
        if ev.url:
            entry += f"[More Info]({ev.url})\n"
        lines.append(entry.strip())
    return "Weekend events upcoming:\n\n" + "\n\n".join(lines)

In [278]:
markdown_output = events_to_markdown(deduped_events.event_list)
print(markdown_output)

Weekend events upcoming:

1. Propnex Family Zone Outdoor Playground at Gardens by the Bay*
📍 Gardens by the Bay, Propnex Family Zone, near Active Garden and Waterfront Plaza
🗓 Till 31 Dec 2025
Free outdoor playground featuring climbing towers, suspension bridges, sand pit, playhouse, market store, and obstacle courses. Beautiful backdrop with Marina Bay Sands.
👶 Age group: Kids
[More Info](https://www.bykido.com/blogs/events-and-activities/top-things-to-do-and-places-to-go-with-kids-this-weekend-in-singapore?srsltid=AfmBOoqWgLrta2lAmmYi4BQgBgvgV9FpmrQuQOMpZkL6MmtKwtiwps5G)

2. *The Great Reef Rescue*: A Fun & Free Conservation Adventure for Families*
📍 Not specified in the content provided
🗓 06/09/2025 and 07/09/2025
A free conservation adventure that encourages families to participate in a fun activity aimed at raising awareness about reef conservation.
👶 Age group: All ages
[More Info](https://www.sunnycitykids.com/blog/kids-activities-this-week-in-singapore)

3. Ranger Buddies: Miss

In [279]:
def escape_markdown(text: str) -> str:
    # Escape all Telegram MarkdownV2 special chars
    return re.sub(r'([_*\[\]()~`>#+\-=|{}.!])', r'\\\1', text)

def send_events_to_telegram(content, bot_token: str, chat_id: str):

    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {
        "chat_id": chat_id,
        "text": escape_markdown(content),
        "parse_mode": "MarkdownV2"  # enables bold, links, emojis, etc.
    }

    resp = requests.post(url, data=payload)
    return resp.json()

In [280]:
send_events_to_telegram(markdown_output, os.getenv("TELEGRAM_BOT_TOKEN"), os.getenv("TELEGRAM_CHAT_ID"))

{'ok': True,
 'result': {'message_id': 5,
  'sender_chat': {'id': -1003097621087,
   'title': 'SGKidEvents',
   'username': 'sgkidevents',
   'type': 'channel'},
  'chat': {'id': -1003097621087,
   'title': 'SGKidEvents',
   'username': 'sgkidevents',
   'type': 'channel'},
  'date': 1757082407,
  'text': 'Weekend events upcoming:\n\n1. Propnex Family Zone Outdoor Playground at Gardens by the Bay*\n📍 Gardens by the Bay, Propnex Family Zone, near Active Garden and Waterfront Plaza\n🗓 Till 31 Dec 2025\nFree outdoor playground featuring climbing towers, suspension bridges, sand pit, playhouse, market store, and obstacle courses. Beautiful backdrop with Marina Bay Sands.\n👶 Age group: Kids\n[More Info](https://www.bykido.com/blogs/events-and-activities/top-things-to-do-and-places-to-go-with-kids-this-weekend-in-singapore?srsltid=AfmBOoqWgLrta2lAmmYi4BQgBgvgV9FpmrQuQOMpZkL6MmtKwtiwps5G)\n\n2. *The Great Reef Rescue*: A Fun & Free Conservation Adventure for Families*\n📍 Not specified in the 